# FEVER on Google Colab
Start with a GPU setup check. Then run **RoBERTa, Qwen or Llama** against the full **19,998-claim official FEVER development set**, with resumable outputs in Google Drive.

**This notebook contains no new results.** The included small training/validation sets only test the software. They are not final fine-tuning experiments. This benchmark has previously been inspected; do not call it an untouched test set.

1. Open this file in [Google Colab](https://colab.research.google.com/), then select **Runtime → Change runtime type → T4 GPU**.
2. Upload `FEVER_Colab_Bundle.zip` to your Google Drive (the bundle includes the existing RoBERTa checkpoint, full evaluation inputs, source code and tiny training pilots).
3. Run sections 1–5. They verify the GPU, uploaded data and source hashes. They do not start training or the full evaluation.
4. Read the model download estimate, enable the download checkbox, and run the Qwen smoke check.
5. Enable a long run only after inspecting that check. Free Colab sessions may end early. Drive checkpoints allow resuming, but free Colab does **not** guarantee background execution after closing your laptop.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [Qwen 3B](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), [Llama 3.2 3B](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).

## 1. Check your GPU
This prints the allocated GPU and available memory. It stops if the runtime is CPU-only.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

## 2. Install the Colab dependencies
This installs Python libraries in the **Colab runtime**, not on your laptop. It does not replace Colab's CUDA-enabled PyTorch. Model weights download later, after the size estimate.

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

## 3. Connect Drive and select the bundle
Mounting Drive asks you to authenticate. The notebook never prints account tokens. Keep the bundle private; it does not need to be published anywhere.

Change `BUNDLE_PATH` if you placed the ZIP in another folder. The working extraction is temporary; experiment results are saved separately in Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
BUNDLE_PATH = "/content/drive/MyDrive/FEVER_Colab_Bundle.zip" # @param {type:"string"}
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
archive = Path(BUNDLE_PATH)
if not archive.is_file():
    raise FileNotFoundError("Upload FEVER_Colab_Bundle.zip to Drive, then correct BUNDLE_PATH above.")
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)

## 4. Experiment settings
These settings are recorded and checked on resume. Changing the model, prompt, token budget, source code or package/GPU identity requires a **new experiment name**.

- **Qwen:** Qwen2.5-3B-Instruct in 4-bit NF4, the first setup test.
- **Llama:** Llama-3.2-3B-Instruct in 4-bit NF4. You must obtain access on its model page yourself and add `HF_TOKEN` in Colab Secrets. This is a new 3B model, **not** the earlier Ollama Llama 3.1 8B.
- **RoBERTa:** the exact existing project checkpoint, with 512-token inputs.
- Only one classifier is loaded at a time. RAG contexts are shared between conditions; different tokenizers and RoBERTa's shorter context remain limitations.
- This HF generator uses ordinary greedy decoding, not Ollama's JSON-schema constrained decoder. Invalid outputs are retained as errors. The 512-token output cap is a declared new setting.

In [ ]:
EXPERIMENT_NAME = "colab_full_dev_v1" # @param {type:"string"}
MODEL_FAMILY = "qwen" # @param ["qwen", "llama", "roberta"]
ALLOW_MODEL_DOWNLOADS = False # @param {type:"boolean"}
SESSION_MINUTES = 45 # @param {type:"integer"}
MAX_NEW_TOKENS = 512 # @param {type:"integer"}
ROBERTA_MAX_LENGTH = 512 # @param [256, 512] {type:"raw"}
# Leave empty for the included original checkpoint. A completed fine-tuned model may be selected later.
ROBERTA_CHECKPOINT_OVERRIDE = "" # @param {type:"string"}
RUN_ROOT = Path("/content/drive/MyDrive/FEVER_runs") / EXPERIMENT_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if not 1 <= SESSION_MINUTES <= 300:
    raise ValueError("Use a bounded session between 1 and 300 minutes.")
if not 1 <= MAX_NEW_TOKENS <= 1024:
    raise ValueError("Use 1–1024 output tokens; freeze the choice before the full evaluation.")
if ROBERTA_MAX_LENGTH not in (256, 512):
    raise ValueError("RoBERTa supports 256 or 512 here, not arbitrarily long article input.")
print("Results saved to:", RUN_ROOT)

## 5. Check model access and report download sizes
This reads model metadata; it does not download model weights. The listed weight sizes are disk/network downloads **before** runtime quantization. Four-bit loading lowers GPU memory requirements, not necessarily network transfer.

If Llama access fails, use Qwen for the setup check while requesting Llama access separately. No token is written to the experiment protocol.

In [ ]:
EMBEDDING = fever.remote_metadata("sentence-transformers/all-MiniLM-L6-v2")
if MODEL_FAMILY == "roberta":
    MODEL_METADATA = {"path": ROBERTA_CHECKPOINT_OVERRIDE or str(PROJECT / "checkpoint/roberta_evidence")}
    CONDITION = "roberta_" + ("selected_checkpoint" if ROBERTA_CHECKPOINT_OVERRIDE else "existing") + "_rag_" + str(ROBERTA_MAX_LENGTH)
    print("RoBERTa uses the uploaded checkpoint; no new classifier weights download.")
else:
    MODEL_METADATA = fever.remote_metadata(fever.MODELS[MODEL_FAMILY])
    CONDITION = MODEL_FAMILY + "_3b_nf4_parts"
print("Condition:", CONDITION)
print("Full evaluation target:", len(EVAL_ROWS), "claims. No evaluation has started.")

## 6. Run one engineering check first
Enable `ALLOW_MODEL_DOWNLOADS` in section 4 only after reading section 5. This check uses **one training-pilot claim**, not a reduced test set. Its article URLs come from that example's retrieved pages; full evaluation uses the exported original top-ten ranking.

Inspect the full prompt, actual article passages and raw output. A successful check verifies execution, not model accuracy. Nothing from this check is pooled into the 19,998-claim result.

In [ ]:
RUN_SMOKE_CHECK = False # @param {type:"boolean"}
if not RUN_SMOKE_CHECK:
    print("Not started. Enable RUN_SMOKE_CHECK after setup.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Read section 5, enable ALLOW_MODEL_DOWNLOADS, rerun section 4 and section 5, then rerun this cell.")
else:
    from urllib.parse import quote
    import time
    example = fever.read_rows(PROJECT / "data/train_pilot.jsonl")[0]
    pages = list(dict.fromkeys(e["page_id"] for e in example["evidence"]))
    from fetch_wikipedia_articles import page_url
    example = {"id": example["id"], "claim": example["claim"],
               "ranked_articles": [{"rank": i + 1, "url": page_url(p)} for i, p in enumerate(pages)]}
    encoder = fever.ResidentEmbedder(EMBEDDING)
    verifier = None
    try:
        context = fever.context_for(example, PROJECT, RUN_ROOT / "software_check", encoder, EMBEDDING)
        print("Claim:", example["claim"])
        print("Evidence status:", context["status"])
        if context["status"] != "READY":
            raise RuntimeError("Wikipedia preparation failed. Inspect saved error before any full run: " + str(context.get("error")))
        verifier = fever.RobertaVerifier(MODEL_METADATA["path"], ROBERTA_MAX_LENGTH) if MODEL_FAMILY == "roberta" else fever.CausalVerifier(MODEL_METADATA, PROJECT)
        result = verifier.predict(example, context, max_new_tokens=MAX_NEW_TOKENS)
        fever.write_once(RUN_ROOT / "software_check" / (CONDITION + "_" + str(time.time_ns()) + ".json"), result)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("Supplied evidence:", json.dumps(context["supplied_evidence"], indent=2, ensure_ascii=False))
    finally:
        if verifier is not None:
            verifier.close()
        del encoder
        fever.release()

## 7. Run / resume the full 19,998-claim evaluation
This is the long operation. It is **off by default**. It attempts the whole official set in its original order and pauses after the session budget. Rerun this cell to resume; do not delete the Drive folder.

Run one model family at a time, then change `MODEL_FAMILY` in section 4 and rerun sections 4–5 and this section. Each family reuses the same saved per-claim RAG passages. Newly fetched full articles and vectors are deleted after each claim; supplied passages, revisions and exact model calls remain.

Failed evidence preparation is an **INVALID execution**, not NEI. Completed invalid predictions remain errors; there is no silent retry. No strict FEVER evidence score is claimed for live article passages. Free Colab may stop early, and GPU availability is not guaranteed.

In [ ]:
RUN_FULL_EVALUATION = False # @param {type:"boolean"}
if not RUN_FULL_EVALUATION:
    print("Full run is off. Target remains all 19,998 claims.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Model downloads are disabled; inspect section 5 before enabling them.")
else:
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, CONDITION, MODEL_FAMILY,
                        MODEL_METADATA, EMBEDDING, minutes=SESSION_MINUTES,
                        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH)

## 8. Check progress and score complete conditions
Partial runs show progress only. A final accuracy table is produced only when a condition covers **all 19,998 IDs**. INVALID remains in the denominator. Metrics are label accuracy, macro-F1, per-class precision/recall/F1 and confusion matrices.

A complete official-development result is still a development result. The benchmark has already been inspected. The historical audit found three normalized-text overlaps with the original checkpoint’s training pool; all 19,998 benchmark rows remain in evaluation. This is not an untouched final test.

In [ ]:
conditions = sorted(p.name for p in (RUN_ROOT / "predictions").glob("*") if p.is_dir())
for name in conditions:
    folder = RUN_ROOT / "predictions" / name
    n = sum(p.name != "protocol.json" for p in folder.glob("*.json"))
    print(name, n, "/", len(EVAL_ROWS))
METRICS = fever.score(PROJECT, RUN_ROOT, conditions, require_full=True)
if METRICS:
    import pandas as pd
    display(pd.DataFrame([{"System": name, "Claims": m["n"], "Accuracy (%)": 100*m["accuracy"],
                           "Macro-F1": m["macro_f1"], "Invalid": m["invalid"]} for name, m in METRICS.items()]))
    import matplotlib.pyplot as plt
    for name, m in METRICS.items():
        display(pd.DataFrame([{"Class": label, "Precision (%)": 100*v["precision"],
                              "Recall (%)": 100*v["recall"], "F1 (%)": 100*v["f1"]}
                             for label, v in m["per_class"].items()]))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.imshow(m["confusion_matrix"], cmap="Blues")
        ax.set_xticks(range(4), ["SUPPORTS", "REFUTES", "NEI", "INVALID"])
        ax.set_yticks(range(3), ["SUPPORTS", "REFUTES", "NEI"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name + " — all 19,998 claims")
        for i, row in enumerate(m["confusion_matrix"]):
            for j, value in enumerate(row): ax.text(j, i, str(value), ha="center", va="center")
        plt.show()

## 9. Optional RoBERTa layer-unfreezing check
This is a **training operation**, off by default. Run it only when you choose to start training.

- `pilot`: included **96 training + 96 validation examples**, used only to verify that training and checkpoints work.
- `full`: requires `train_full.jsonl` and `validation_full.jsonl` from the optional full-training export. The starter bundle intentionally does not prepare this hours-long local retrieval job.
- All conditions start from the **same original task-trained RoBERTa checkpoint**. This is continued fine-tuning, not starting from raw pretrained RoBERTa.
- Freeze all weights, then unfreeze the classification head and the last **1, 2 or 3 Transformer blocks**. Earlier blocks remain frozen.
- Fixed seed 42, 2 epochs, learning rate 2e-5, batch size 4, gradient accumulation 4, maximum 512 tokens. Validation selects the checkpoint; official development labels never train or select it.
- Training uses historical retrieved sentence text for every class, not the new live-article RAG corpus. Report that distribution difference. The pilot includes nonempty NEI evidence to avoid the old empty-input shortcut.

Change only the number of unfrozen layers between controlled conditions. Llama/Qwen here are inference baselines: **this notebook does not implement their fine-tuning**. Quantized LLM adapter training would be a separate experiment and is not equivalent to unfreezing their original last layers.

In [ ]:
RUN_ROBERTA_TRAINING = False # @param {type:"boolean"}
UNFREEZE_LAST_N = 1 # @param [1, 2, 3] {type:"raw"}
TRAIN_SCOPE = "pilot" # @param ["pilot", "full"]
if not RUN_ROBERTA_TRAINING:
    print("Training is off. Pilot training is a software check, not a completed research experiment.")
else:
    final_checkpoint = fever.finetune_roberta(PROJECT, RUN_ROOT,
                    last_layers=UNFREEZE_LAST_N, scope=TRAIN_SCOPE, minutes=SESSION_MINUTES)
    if final_checkpoint:
        print("To evaluate this model, set ROBERTA_CHECKPOINT_OVERRIDE in section 4 to:")
        print(str(final_checkpoint))
        print("Use a distinct condition name for each trained checkpoint; see the next cell.")

### Evaluate a completed fine-tuned checkpoint
Use the full 19,998 inputs again. The condition name includes its layer count and training scope. A pilot-trained condition is explicitly named pilot and must not be called full-training performance.

In [ ]:
RUN_FINETUNED_EVALUATION = False # @param {type:"boolean"}
if not RUN_FINETUNED_EVALUATION:
    print("Fine-tuned evaluation is off.")
else:
    trained = RUN_ROOT / "training" / ("roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N))
    if not (trained / "complete.json").is_file():
        raise RuntimeError("Training has not completed. Resume training before evaluating its final checkpoint.")
    name = "roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N) + "_rag_" + str(ROBERTA_MAX_LENGTH)
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, name, "roberta",
        {"path": str(trained / "final")}, EMBEDDING, minutes=SESSION_MINUTES,
        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH)

## What this notebook does not claim
- No GPU run has been executed by the notebook author on your Colab account. Local tests cover data guards, response validation and layer selection, not cloud throughput or final accuracy.
- The 3B NF4 models and HF decoder differ from the earlier 8B Ollama experiment. New results need their own names and must not replace the old results.
- The three model families have different architectures and effective context budgets. Same passages do not imply identical retained evidence.
- This does not implement a new symbolic architecture or a Reddit study. Your professor's intended Reddit task still needs clarification.
- Free Colab is time-limited. Save Drive outputs, resume batches, and measure throughput before estimating a full-run completion date. Do not use keep-alive tricks to bypass service limits.

All source code used by a condition is in the uploaded bundle under `src/`, with SHA-256 hashes in `manifest.json`. Exact prompts, responses, retained inputs and evidence revisions are saved under your Drive run folder.